In [1]:
import gzip
import pickle
import pandas as pd

In [2]:
from matbench_discovery.data import DataFiles
with gzip.open(DataFiles.mp_patched_phase_diagram.mp_patched_phase_diagram.path, mode="rb") as zip_file:
    ppd_mp = pickle.load(zip_file)

In [3]:
mp_20_csv_data = pd.read_csv("mp_20_test.csv.gz", index_col="material_id")
computed_data = pd.read_csv("mp_20_PBE-6.csv.gz", index_col="material_id")

In [4]:
from mp_api.client import MPRester
from dotenv import find_dotenv, get_key
MP_API_KEY = get_key(find_dotenv(), "MP_API_KEY")

In [9]:
with MPRester(MP_API_KEY) as mpr:
    for mp_id in computed_data.index:
        # reference_records = mpr.get_entry_by_material_id(mp_id)
        #for record in reference_records:
        #    found = False
        #    if record.data['run_type'] in ("GGA", "GGA_U"):
        #        computed_data.loc[mp_id, "mp_e_above_hull_from_our_ppd"] = ppd_mp.get_e_above_hull(record, allow_negative=True)
        #        computed_data.loc[mp_id, "mp_e_uncorrected"] = record.uncorrected_energy
        #        if found:
        #            raise ValueError(f"Multiple records found for {mp_id}")
        #        found = True
        thermo_docs = mpr.materials.thermo.search(
            material_ids=[mp_id], thermo_types=["GGA_GGA+U"])
        if len(thermo_docs) > 1:
            raise ValueError(f"Multiple records found for {mp_id}")
        computed_data.loc[mp_id, "mp_e_above_hull_from_db"] = thermo_docs[0].energy_above_hull
        assert len(thermo_docs[0].entries) == 1
        record = next(iter(thermo_docs[0].entries.values()))
        computed_data.loc[mp_id, "mp_e_above_hull_from_our_ppd"] = ppd_mp.get_e_above_hull(record, allow_negative=True)
        computed_data.loc[mp_id, "mp_e_uncorrected"] = record.uncorrected_energy

Retrieving ThermoDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
from pymatgen.entries.computed_entries import ComputedEntry
from ast import literal_eval
computed_data["PBE_54_energy_uncorrected"] = computed_data.entries.map(
    lambda x: ComputedEntry.from_dict(literal_eval(x)).uncorrected_energy)
computed_data.loc["mp-19449", "PBE_54_energy_uncorrected_script"] = -82.51573047
computed_data.loc["mp-752737", "PBE_54_energy_uncorrected_script"] = -77.40705622
computed_data.drop(columns=["entries", "structure"])

,e_above_hull_corrected,e_uncorrected,mp_e_above_hull_from_our_ppd,mp_e_uncorrected,mp_e_above_hull_from_db,PBE_54_energy_uncorrected,PBE_54_energy_uncorrected_script
material_id,,,,,,,
mp-1030119,0.002284,-90.780912,2.072728e-03,-90.783443,0.002073,-90.780912,NaN
mp-19449,0.000146,-83.156814,-6.217249e-15,-83.158863,0.000000,-83.156814,-82.515730
mp-28599,-0.000429,-23.586253,2.612457e-04,-23.581418,0.000261,-23.586253,NaN
mp-864930,-0.005719,-30.145068,0.000000e+00,-30.122192,0.000000,-30.145068,NaN
mp-1103947,0.000135,-62.121757,2.610386e-04,-62.119994,0.000261,-62.121757,NaN
mp-752737,0.068126,-78.509810,6.763063e-02,-78.515757,0.067631,-78.509810,-77.407056
mp-1220086,0.020144,-92.493216,1.904982e-02,-92.506348,0.019050,-92.493216,NaN
mp-1221692,0.001701,-56.891229,0.000000e+00,-56.911640,0.000000,-56.891229,NaN
mp-1221948,0.044604,-31.108918,4.455771e-02,-31.109103,0.044558,-31.108918,NaN


In [18]:
mp_entry = record

In [19]:
from pymatgen.entries.computed_entries import ComputedEntry
from ast import literal_eval
our_entry = ComputedEntry.from_dict(literal_eval(computed_data.loc["mp-1516349"].entries))

In [43]:
mp_entry.parameters['potcar_spec'][1]

{'titel': 'PAW_PBE Eu 08Apr2002',
 'hash': 'd466d046adf21f6146ee9644049ea268',
 'summary_stats': None}

In [42]:
our_entry.parameters['potcar_spec'][1]

{'titel': 'PAW_PBE Eu 23Dec2003',
 'hash': '859fd9e4dcfab950524fa81bb600e9b4',
 'summary_stats': {'keywords': {'header': ['dexc',
    'eatom',
    'eaug',
    'enmax',
    'enmin',
    'iunscr',
    'lcor',
    'lexch',
    'lpaw',
    'lultra',
    'ndata',
    'orbitaldescriptions',
    'pomass',
    'qcut',
    'qgam',
    'raug',
    'rcloc',
    'rcore',
    'rdep',
    'rmax',
    'rpacor',
    'rrkj',
    'rwigs',
    'step',
    'titel',
    'vrhfin',
    'zval'],
   'data': ['localpart',
    'gradientcorrectionsusedforxc',
    'corecharge-density(partial)',
    'atomicpseudocharge-density',
    'nonlocalpart',
    'reciprocalspacepart',
    'realspacepart',
    'reciprocalspacepart',
    'realspacepart',
    'nonlocalpart',
    'reciprocalspacepart',
    'realspacepart',
    'reciprocalspacepart',
    'realspacepart',
    'nonlocalpart',
    'reciprocalspacepart',
    'realspacepart',
    'reciprocalspacepart',
    'realspacepart',
    'nonlocalpart',
    'reciprocalspacepart'